In [1]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

import dotenv
load_dotenv()
model=init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    profile={"max_input_tokens":128_000}
)


In [3]:
#humanIntheLoopMiddleware
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent=create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
      interrupt_on={
          "get_weather": True,
          "get_news": True,
          "read_email_tool": False,
          "send_email_tool": {"allowed_decisions":["approve","reject"],"description":"发送邮件中断了！"},
      },
        description_prefix="中断啦~"
    ),
                ]
)
config={"configurable":{"thread_id":"1"}}
response=agent.invoke({"messages": [HumanMessage(content="请帮我查询今天北京的天气"
                                      "查询今日新闻"
                                      "查看ID为 'sk2131421' 的邮件内容，"
                                      "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                                      "同时做这四件事")]
},config=config)
rprint(response)

{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='763c1717-660d-4d5a-954a-4cf4d68a6c89'
        ),
        AIMessage(
            content='我来同时为您处理这四件事：查询北京天气、查询今日新闻、读取邮件以及发送邮件。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': "The user wants me to do four things simultaneously:\n1. Query today's weather
in Beijing\n2. Query today's news\n3. Read email with ID 'sk2131421'\n4. Send email to 15641685664@qq.com with 
subject '哈哈哈' and body '你好啊'\n\nThese are all independent, so I can do them all in parallel."
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 265,
                    'prompt_tokens': 595,
                    'total_tokens': 860,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 76,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 83
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '6ba75c24-9635-4883-9eec-88f7675998b0',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a06903-e676-7620-a625-b22392f4d304-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_OH8RH8h7qjxPJHLr1jHt8936',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_b9TFpsHSCMSXSQL1WfPL6543', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_Up4j6rcMoDrnkpcNDB4H2448',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_FqmopHXC86WZtsCdJe0I0332',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 595,
                'output_tokens': 265,
                'total_tokens': 860,
                'input_token_details': {'cache_read': 512},
                'output_token_details': {'reasoning': 76}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京'},
                        'description': "中断啦~\n\nTool: get_weather\nArgs: {'city': '北京'}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦~\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断了！'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'get_news', 'allowed_decisions': ['a

In [11]:
#举例的过程2

weather_decisions={
    "type":"edit",
    "edited_action":{
        "name":"get_weather",
        "args":{"city":"上海","is_forcast":True},
    }
}
news_decisions={
    "type":"approve",
}
send_email_decision={
    "type":"approve",
}

decisions={
    "decisions":[]
}
interrupts=response.get("__interrupt__",[])
action_requests=interrupts[0].value["action_requests"]
for action in  action_requests:
    if action["name"]=="get_weather":
        decisions["decisions"].append(weather_decisions)
    if action["name"]=="get_news":
        decisions["decisions"].append(news_decisions)
    if action["name"]=="send_email_tool":
        decisions["decisions"].append(send_email_decision)
if interrupts :
    resumed_response=agent.invoke(
        Command(resume=decisions),
        config=config
    )
    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我来同时为您处理这四件事：查询北京天气、查询今日新闻、读取邮件以及发送邮件。
Tool Calls:
  get_weather (call_00_OH8RH8h7qjxPJHLr1jHt8936)
 Call ID: call_00_OH8RH8h7qjxPJHLr1jHt8936
  Args:
    city: 上海
    is_forcast: True
  get_news (call_01_b9TFpsHSCMSXSQL1WfPL6543)
 Call ID: call_01_b9TFpsHSCMSXSQL1WfPL6543
  Args:
  read_email_tool (call_02_Up4j6rcMoDrnkpcNDB4H2448)
 Call ID: call_02_Up4j6rcMoDrnkpcNDB4H2448
  Args:
    email_id: sk2131421
  send_email_tool (call_03_FqmopHXC86WZtsCdJe0I0332)
 Call ID: call_03_FqmopHXC86WZtsCdJe0I0332
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

上海今天天气不错
明天下雨
================================= T